<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day05_practice1_%EC%86%8C%ED%94%84%ED%8A%B8%EB%A7%A5%EC%8A%A4_%ED%95%B4%EB%B6%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [5차시 실습 1] 소프트맥스와 크로스 엔트로피

import torch
import torch.nn as nn

torch.manual_seed(42)

In [ ]:
# 셀 1. 문제 상황 - 이진에서 다중으로

# 모델의 마지막 층이 내놓는 '날 점수'를 로짓(logit)
logits = torch.tensor([2.0, 1.0, 0.1]) # [포트홀, 균열, 맨홀] 점수]
print("로짓(날 점수):'):", logits.tolist()) #

로짓(날 점수):'): [2.0, 1.0, 0.10000000149011612]


In [ ]:
# 셀 2. 소프트맥스 - 점수를 확률로

# softmax(z_i) = e^{z_i} / Σ e^{z_j}
#   ① e^z : 지수함수 적용, 전부 양수로 만들고, 큰 점수를 '더' 크게 (지수의 힘)
#   ② ÷합 : 전체 합이 1 이 되게 → 확률

exp = torch.exp(logits) # 지수 함수를 적용, 모든 값이 양수가 되고, 큰 점수는 더 큰 가중치를 부여
manual_softmax = exp / exp.sum() # 모든 요소의 합계로 나눈다. 모든 결과값의 합이 1, 각 클래스에 대한 확률 분포를 얻게 된다

print("수동 소프트맥스:" , [round(v, 4) for v in manual_softmax.tolist()])
print("합계:", manual_softmax.sum().item())

# PyTorch 함수와 비교
torch_softmax = torch.softmax(logits, dim=0)
#dim=0은 계산이 텐서의 첫 번째 차원을 따라 이루어짐.
# 1차원이면 모든 요소에 대해 softmax를 계산하여 합이 1이 되게
# 2차원이면 dim=0 행 축을 없앰 -> 위아래(행)를 합침 -> 열별 합, 그 결과로 하나의 소프트맥스를 계산
assert torch.allclose(manual_softmax, torch_softmax) # torch.allclose() 두 텐서의 모든 원소가 오차 범위 내에서 같으면 True
# assert 조건이 True면 아무 일 없이 통과하고, False면 AssertionError를 발생시켜 실행을 멈춘다

수동 소프트맥스: [0.659, 0.2424, 0.0986]
합계: 0.9999999403953552


In [ ]:
# 셀 3. 시그모이드와의 관계
# 클래스가 딱 2개일 때 소프트맥스는 시그모이드와 같은 답은 낸다.

z = torch.tensor([1.5, 0.0]) # [클래스1 점수, 클래스0(음) 점수]
print("\n2클래스 소프트맥스:", torch.softmax(z, dim=0)[0].item())
print("시그모이드(1.5)  :", torch.sigmoid(torch.tensor(1.5)).item()) # 확률값


2클래스 소프트맥스: 0.8175744414329529
시그모이드(1.5)  : 0.8175744414329529


In [ ]:
# 셀 4. 원-핫 인코딩 - 정답의 표현법
# 숫자 라벨: 1
# 원-핫     :  [0, 1, 0]

y_label = torch.tensor(1) # 숫자 라벨
y_onehot = nn.functional.one_hot(y_label, num_classes=3) # 숫자 라벨을 길이 3짜리 원-핫 벡터로 바꾼다
print("\n원-핫:", y_onehot.tolist())


원-핫: [0, 1, 0]


In [ ]:
# 셀 5. 크로스 앤트로피 - 다중분류의 손실
# 확률에 -log를 씌운 값, 정답에 높은 확률을 줬으면 손실이 작고, 낮은 확률을 줬으면 손실이 커진다

probs = manual_softmax #[0.659, 0.2424, 0.0986] # [포트홀, 균열, 맨홀] 확률값
answer = 1              #정답: 균열

ce_manual = -torch.log(probs[answer])
print(f"\n정답(균열)에 준 확률: {probs[answer]:.4f}")
print(f"수동 CE = -log(0.24) = {ce_manual:.4f}")

# PyTorch 의 CrossEntropyLoss 로 같은 값 확인
loss_fn = nn.CrossEntropyLoss()
ce_torch = loss_fn(logits.unsqueeze(0), torch.tensor([answer])) # logits.unsqueeze(0) : (3.) -> (1,3) (배치, 클래스)
print(f"CrossEntropyLoss  = {ce_torch:.4f}")
assert torch.allclose(ce_manual, ce_torch, atol=1e-4)
print(" 일치 - CrossEntropyLoss = Softmax + (-log) 를 한 번에") # 다중분류는 모델 끝에 Softmax 를 안 붙인 이유


정답(균열)에 준 확률: 0.2424
수동 CE = -log(0.24) = 1.4170
CrossEntropyLoss  = 1.4170
 일치 - CrossEntropyLoss = Softmax + (-log) 를 한 번에


In [ ]:
# 셀 6. argmax - 확률에서 최종 답으로

batch_logits = torch.tensor([[2.0, 1.0, 0.1],    # 사진 1 , 각 열이 클래스(포트홀/ 균열/ 맨홀) 점수
                             [0.2, 0.4, 3.0],    # 사진 2
                             [1.0, 2.5, 0.3]])  # 사진 3

classes = ["포트홀", "균열", "맨홀"]

preds = batch_logits.argmax(dim=1) # dim=1은 열 축을 없애며 각 행에서 가장 큰 값의 위치(인덱스)를 찾는다
probs = torch.softmax(batch_logits, dim=1) # 각 행(사진)별로 로짓을 확률로 변환, 행마다 확률 합이 1
print("preds:", preds)
print("probs:\n", probs)

for i, p in enumerate(preds): # (0,0), (1,2), (2,1) # [포트홀, 균열, 맨홀]
  print(f"사진{i+1}: {classes[p]} ({probs[i, p]:.0%})") # 소수점 없이 %로

  # "type": "포트홀", "confidence": 0.92
  # 로짓->소프트맥스->argmax

preds: tensor([0, 2, 1])
probs:
 tensor([[0.6590, 0.2424, 0.0986],
        [0.0536, 0.0654, 0.8810],
        [0.1673, 0.7497, 0.0831]])
사진1: 포트홀 (66%)
사진2: 맨홀 (88%)
사진3: 균열 (75%)
